# Grokipedia analysis (cleaned)

Carry out the keyword analysis on our parsed Grokipedia data (parsed using `parse_grokipedia.py`)

Authorship:
- Alex Diaz-Papkovich

## Import libraries and functions

In [4]:
import numpy as np
import os
import pandas as pd
import pickle
import json
import re

from collections import defaultdict

In [5]:
# keywords used to detect genetics text/sections
kw_patterns = {
    'admix':re.compile(r'admix',re.IGNORECASE),
    'autosom':re.compile(r'autosom',re.IGNORECASE),
    'biobank':re.compile(r'biobank',re.IGNORECASE),
    'chromosom':re.compile(r'chromosom',re.IGNORECASE),
    'DNA':re.compile(r'\bDNA\b'),
    'genes':re.compile(r'\bgenes?\b', re.IGNORECASE),
    'genetic':re.compile(r'genetic',re.IGNORECASE),
    'genom':re.compile(r'genom',re.IGNORECASE),
    'genotyp':re.compile(r'genotyp',re.IGNORECASE),
    'haplo':re.compile(r'haplo',re.IGNORECASE),
    'mitochon':re.compile(r'mitochon',re.IGNORECASE),
    'mt-DNA':re.compile(r'mt-DNA|mtDNA',re.IGNORECASE),
    'PCA':re.compile(r'PCA|(?i:principal component)'),
    'x chromosom':re.compile(r'x chromosom|x-chromosom',re.IGNORECASE),
    'y chromosom':re.compile(r'y chromosom|y-chromosom',re.IGNORECASE),
    'y-DNA':re.compile(r'y-DNA|yDNA',re.IGNORECASE),
}

In [6]:
def multiple_replace(replacements, text):
    """
    Use regex to translate multiple keys from a dictionary to their values.
    We use this to substitute the reference tags in the page for their title and URL.

    replacements: dict where (key,value) pairs are the ref tag and its value e.g. ("[1]","Google -- Google.com")
    text: text for the replacement

    Output: substituted values
    """
    # Create a regular expression from the dictionary keys
    regex = re.compile("(%s)" % "|".join(map(re.escape, replacements.keys())))
    # For each match, look-up corresponding value in dictionary
    return regex.sub(lambda mo: replacements[mo.group()], text) 

## Single page analysis

In [8]:
parsed_dir = "data/grokipedia/parsed"

In [9]:
page = "Black_people"

In [10]:
in_file = os.path.join(parsed_dir, page + ".json")

In [11]:
with open(in_file, "r") as f:
    in_data = json.loads(f.read().strip())

f.close()

In [214]:
# Create a dictionary for the references (so we can substitute in the text and run the keyword parser on it)
ref_dict = {}
for ref in in_data["references"]:
    ref_dict["[" + ref["id"].split("ref-")[1] + "]"] = "<ref>" + ref["title"] + "; URL: " + ref["url"] + "</ref>"

In [215]:
# Create some convenience data for ourselves

for section in in_data["sections"]:
    # Create numeric header levels (and set intro to 0 to match wikipedia)
    if section["header"]=="(Introduction)":
        section["level_numeric"] = 0
    else:
        section["level_numeric"] = int(section["level"].split("h")[1].strip())

    # Plain text (strip out references)
    section["plain_text"] = re.sub(r'\[\d+\]','',section["text"])

    # Substitute in the reference values for each ref tag
    tags = re.findall(r'\[\d+\]', section["text"])
    section["text_with_refs"] = multiple_replace(ref_dict, section["text"])

### Keyword search

What we'd like to find out:
* How early do we find genetics keywords?
* How much of the text does the genetics section take up?
* How many total genetics keywords do have?

In [216]:
# Count of all keyword hits
kw_hits = defaultdict(int)

for section in in_data["sections"]:
    search_text = section["text_with_refs"]
    for kw in kw_patterns.keys():
        regex_hits = kw_patterns[kw].findall(search_text)
        kw_hits[kw]+=len(regex_hits)

        

In [218]:
# total hits:
print("Total keyword hits:", sum(kw_hits.values()))

Total keyword hits: 68


In [17]:
# First instance of a genetics keyword
all_text = "\n".join([section["text_with_refs"] for section in in_data["sections"]])

In [19]:
# Section lengths
section_lengths = list()
for section in in_data["sections"]:
    section_lengths.append(len(section["plain_text"]) + len(section["header"]))

In [25]:
# Detect genetics keywords in section headers and section text
header_flags = list() # Subsections inherit parent section's flag
section_flags = list() # Subsections inherit parent section's flag
subsection_flags = list() # Look at subsections alone (no flag inheritance)

header_flags.append(False) # Intro by default is not a genetics section

highest_level = 99

only_hits = True # Print only subsections where genetics keywords are present in the text

for section in in_data["sections"]:
    if section["level_numeric"]==0:
        continue
        
    level = section["level_numeric"]
    highest_level = min(highest_level, level)

    if level <= highest_level:
        # First section will (by default) set the temp_flag to false
        temp_flag_header = False
        temp_flag_section = False
        section_title = section["header"]

    temp_flag_subsection = False # subsections do not inherit flags

    regex_hits_header_all = []
    regex_hits_text_all = []
    
    for kw in kw_patterns.keys():
        regex_hits_header = kw_patterns[kw].findall(section["header"])
        if regex_hits_header:
            temp_flag_header = True
            regex_hits_header_all+=regex_hits_header

        regex_hits_text = kw_patterns[kw].findall(section["text"] + " " + section["header"])
        if regex_hits_text:
            temp_flag_section = True
            temp_flag_subsection = True
            regex_hits_text_all+=regex_hits_text
        

    # Append flags to lists
    header_flags.append(temp_flag_header)
    section_flags.append(temp_flag_section)
    subsection_flags.append(temp_flag_subsection)

    # Print results
    print_regex_header = ""
    print_regex_text = ""
    print_header = ""

    if level==2:
        print_header = "SECTION: " + section_title
    else:
        print_header = "SECTION: " +section_title + "; SUBSECTION: " + section["header"]
    
    if regex_hits_header_all:
        print_regex_header = regex_hits_header_all
    
    if regex_hits_text_all:
        print_regex_text = regex_hits_text_all
        

    print_str = print_header + "\n----Keyword in header? " + str(temp_flag_header) + " " + str(print_regex_header) + \
    "\n----Keyword in subsection? " + str(temp_flag_subsection) + " " + str(print_regex_text)

    if only_hits:
        if regex_hits_header_all or regex_hits_text_all:
            print(print_str)
            print("Number of keywords:", len(print_regex_text))
            print()
    else:
        print(print_str)
        print()

SECTION: Definition and Terminology; SUBSECTION: Modern Classifications and Boundaries
----Keyword in header? False 
----Keyword in subsection? True ['admix', 'admix', 'admix', 'admix', 'admix', 'admix', 'admix', 'DNA', 'gene', 'Genetic', 'genetic', 'genetic', 'genetic', 'genetic', 'genetic', 'principal component']
Number of keywords: 16

SECTION: Genetic and Biological Foundations
----Keyword in header? True ['Genetic']
----Keyword in subsection? True ['Genetic']
Number of keywords: 1

SECTION: Genetic and Biological Foundations; SUBSECTION: Origins and Genetic Diversity
----Keyword in header? True ['Genetic']
----Keyword in subsection? True ['Admix', 'chromosom', 'chromosom', 'DNA', 'gene', 'Genetic', 'genetic', 'genetic', 'genetic', 'genetic', 'Genetic', 'genom', 'genom', 'genom', 'haplo', 'haplo', 'mitochon', 'mtDNA', 'mtDNA', 'principal component', 'Y-chromosom', 'Y-chromosom']
Number of keywords: 22

SECTION: Genetic and Biological Foundations; SUBSECTION: Evolutionary Adaptation

In [477]:
# Other results
page_length = np.sum(np.array(section_lengths)) # Total length
genetics_sections_length = np.sum(np.array(section_lengths)[np.array(header_flags)]) # Genetics section length

print("Genetics sections as proportion of text:", 100*round(genetics_sections_length/page_length,4))

Genetics sections as proportion of text: 7.76


In [478]:
# Adjust for AJ case study (remove two repeated sections)
# sections 5 and 7 might be bugs in the AI-generated text
if page=="Ashkenazi_Jews":
    temp_section_lengths = section_lengths.copy()
    temp_header_flags = header_flags.copy()

    del temp_section_lengths[7]
    del temp_section_lengths[5]
    del temp_header_flags[7]
    del temp_header_flags[5]

    
    page_length = np.sum(np.array(temp_section_lengths)) # Total length
    genetics_sections_length = np.sum(np.array(temp_section_lengths)[np.array(temp_header_flags)]) # Genetics section length
    
    print("Adjusted genetics sections as proportion of text:", 100*round(genetics_sections_length/page_length,4))

In [479]:
# Genetics in intro
intro_text = in_data["sections"][0]["plain_text"]
earliest_term = ""
earliest_posn = 99999

# Position in introduction
for kw in kw_patterns.keys():
    search_result = kw_patterns[kw].search(intro_text)

    if search_result is not None:
        if search_result.start() < earliest_posn:
            print(kw)
            earliest_term = kw
            earliest_posn = search_result.start()

print("Earliest keyword and its position")
print("Term:", earliest_term, "Position:", earliest_posn)
print("Total intro length:", len(intro_text))
print("Position in intro:", 100*round(earliest_posn/len(intro_text),4))

admix
genetic
Earliest keyword and its position
Term: genetic Position: 38
Total intro length: 2627
Position in intro: 1.4500000000000002


## Genetics keywords and sections across all demonyms

In [1]:
parsed_dir = "data/grokipedia/demonyms_202604_parsed/"

In [222]:
genetics_section_flags = []
genetics_keyword_flags = []

for fname in os.listdir(parsed_dir):
    temp_flag_header = False
    temp_flag_kws = False

    if fname.startswith("."):
        continue
    
    with open(os.path.join(parsed_dir, fname), "r") as f:
        in_data = json.loads(f.read().strip())

    f.close()

    for section in in_data["sections"]:
        for kw in kw_patterns.keys():
            regex_hits_header = kw_patterns[kw].findall(section["header"])
            regex_hits_text = kw_patterns[kw].findall(section["text"])
            
            if regex_hits_header:
                temp_flag_header = True
                #regex_hits_header_all+=regex_hits_header
                
            if regex_hits_text:
                temp_flag_kws = True
                #regex_hits_text_all+=regex_hits_text


    genetics_section_flags.append(temp_flag_header)
    genetics_keyword_flags.append(temp_flag_kws)

In [223]:
np.sum(genetics_section_flags)

np.int64(94)

In [224]:
# note: len(os.listdir(parsed_dir)) contains .DStore on this machine so adjust by subtracting 1
np.sum(genetics_section_flags)/(len(os.listdir(parsed_dir))-1)

np.float64(0.7175572519083969)

In [225]:
np.sum(genetics_keyword_flags)

np.int64(120)

In [226]:
np.sum(genetics_keyword_flags)/(len(os.listdir(parsed_dir))-1)

np.float64(0.916030534351145)

## Export plain text

In [12]:
parsed_dir = "data/grokipedia/demonyms_202604_parsed/" # demonyms

In [13]:
out_dir = "data/grokipedia/plain_text"

for fname in os.listdir(parsed_dir):
    if not fname.endswith(".json"):
        continue
        
    with open(os.path.join(parsed_dir, fname), "r") as f:
        in_data = json.loads(f.read().strip())

    f.close()

    text = ""

    for section in in_data["sections"]:
        # Add headers and a line break
        text+= int(section["level"][1])*"=" + section["header"] + int(section["level"][1])*"="
        text+= "\n"
        # Strip citations and add text
        text+= re.sub(r'\[\d+\]','',section["text"]) + "\n"

    if "_output" in fname:
        fout = fname.split("_output")[0]
    elif "_2026" in fname:
        fout = fname.split("_2026")[0]

    fout+=".txt"
    
    with open(os.path.join(out_dir, fout),"w") as fo:
        fo.write(text)
    
    fo.close()

# Export paragraphs containing keywords

Keywords can be in text or URLs

In [48]:
parsed_dir = "demonyms_202604_parsed/"

In [75]:
combined = re.compile(
    r'admix|autosom|biobank|chromosom|\bgenes?\b|genetic|genom|genotyp|haplo|mitochon|mt-DNA|mtDNA|PCA|principal component|x[ -]chromosom|y[ -]chromosom|y-DNA|yDNA',
    re.IGNORECASE
)
dna_pattern = re.compile(r'\bDNA\b')

In [76]:
pghs_with_keywords = list()
pghs_with_keywords_refs = list() # for my reference, check if keywords are in ref tags
pages_with_keywords = list()

for fname in os.listdir(parsed_dir):
    
    if ".json" in fname:
        page_name = fname.split("_2026")[0]
        with open(os.path.join(parsed_dir, fname), "r") as f:
            in_data = json.loads(f.read().strip())
        f.close()

    # Create a dict of references to substitute in
    ref_dict = {}
    for ref in in_data["references"]:
        ref_dict["[" + ref["id"].split("ref-")[1] + "]"] = "<ref>" + ref["title"] + "; URL: " + ref["url"] + "</ref>"

    for section in in_data["sections"]:
        # Create numeric header levels (and set intro to 0 to match wikipedia)
        if section["header"]=="(Introduction)":
            section["level_numeric"] = 0
        else:
            section["level_numeric"] = int(section["level"].split("h")[1].strip())
    
        # Plain text (strip out references)
        section["plain_text"] = re.sub(r'\[\d+\]','',section["text"])
    
        # Substitute in the reference values for each ref tag
        tags = re.findall(r'\[\d+\]', section["text"])
        section["text_with_refs"] = multiple_replace(ref_dict, section["text"])

        pghs_pt = section["plain_text"].split("\n") # plaintext
        pghs = section["text_with_refs"].split("\n") # with citations

        if len(pghs)!=len(pghs_pt):
            print("OH NOES")

        for p in range(0, len(pghs)):
            if combined.search(pghs[p]) or dna_pattern.search(pghs[p]):
                pghs_with_keywords_refs.append(pghs[p]) # text with ref
                pghs_with_keywords.append(pghs_pt[p]) # text without ref
                pages_with_keywords.append(page_name) # page name    

# Grokipedia comparisons to Wikipedia

* Compare lengths of sections (overall)
* Compare counts of genetics terms between pages (original)
* Compare counts of genetics terms between pages (specialized)
* Direct comparisons of genetics sections (when both have them)
  * Length
  * Contents

In [82]:
def sum_genetics_lengths(levels, lengths, idx_genetics, double_count=False):
    """
    Sums all section lengths based on indices. Designed for genetics sections but could be modified.

    Given a page's section levels, section lengths (character counts), and indices of which sections are genetics sections,
    it calculates the total lengths of genetics sections.

    Args:
        levels (list[int]): Hierarchical level of each section. Higher values are subsections. Introduction is always 0.
        lengths (list[int]): List of character counts for each section
        idx_genetics (list[int]): List of indices of sections to count.
        double_count (bool): If False (default), subsections of an already-counted genetics
            section are not counted again. If True, every index in idx_genetics is summed
            independently, regardless of nesting. This was added for the Grokipedia parser.

    Returns:
        int: Summed character count of genetics sections
    """
    if double_count:
        return sum(lengths[i] for i in idx_genetics)

    genetics_set = set(idx_genetics)
    total = 0
    # Track the level of the most recently opened genetics ancestor
    # None means we are not currently inside a genetics section
    active_genetics_level = None

    for i, (level, length) in enumerate(zip(levels, lengths)):
        # If we move to a level that is not deeper than the active genetics ancestor,
        # that ancestor's subtree is over — close it
        if active_genetics_level is not None and level <= active_genetics_level:
            active_genetics_level = None

        if i in genetics_set:
            if active_genetics_level is None:
                # This is the highest-level genetics section in this branch — count it
                total += length
                # Mark this level as an open genetics ancestor
                active_genetics_level = level
            # else: already inside a genetics section, skip (already counted)

    return total

In [83]:
# Plaintext directories
wiki_page_dir = "data/wikipedia_plain_text"
grok_page_dir = "data/grokipedia/plain_text"

## Keyword counts

Count the number of keywords in plaintext.

In [84]:
# Get the list of Grokipedia page names
grok_page_names = [fname.split(".txt")[0] for fname in os.listdir(grok_page_dir) if fname.endswith(".txt")]

In [85]:
# Strategy:
# Import page text
# Loop over regex keywords
# Count in dict with page as keys
# Data to be stored in dict of dicts. First key is page name, second key is keyword

# Grokipedia trackers
grok_kw_counts = defaultdict(dict) # For specific keywords
grok_kw_totals = defaultdict(int) # For total keyword counts

# Wikipedia trackers
wiki_kw_counts = defaultdict(dict)
wiki_kw_totals = defaultdict(int)

for page in grok_page_names:
    # Grokipedia counts
    with open(os.path.join(grok_page_dir, page + ".txt"), "r") as f:
        grok_text = f.read().strip()

    for kw in kw_patterns.keys():
        regex_hits_grok = kw_patterns[kw].findall(grok_text)
        grok_kw_counts[page][kw] = len(regex_hits_grok)
        grok_kw_totals[page]+= len(regex_hits_grok)

    # Wikipedia counts
    with open(os.path.join(wiki_page_dir, page + ".txt"), "r") as f:
        wiki_text = f.read().strip()

    for kw in kw_patterns.keys():
        regex_hits_wiki = kw_patterns[kw].findall(wiki_text)
        wiki_kw_counts[page][kw] = len(regex_hits_wiki)
        wiki_kw_totals[page]+= len(regex_hits_wiki)

In [86]:
# Export results
df1 = pd.DataFrame(
    [("Grokipedia", page, kw, count) for page, kws in grok_kw_counts.items() for kw, count in kws.items()],
    columns=["project","page_name", "keyword", "count"]
)
df2 = pd.DataFrame(
    [("Wikipedia", page, kw, count) for page, kws in wiki_kw_counts.items() for kw, count in kws.items()],
    columns=["project","page_name", "keyword", "count"]
)

In [87]:
df = pd.concat([df1, df2])

In [88]:
#df.to_csv("output/corpus_comparisons/plaintext_keyword_counts.csv", index=False)

## Section comparisons

In [28]:
# Import genetics section information
with open("data/genetics_sections.pkl", "rb") as f:
    output = pickle.load(f)

page_summary          = output["page_summary"]
overall_summary       = output["overall_summary"]
overall_levels        = output["overall_levels"]
overall_titles        = output["overall_titles"]
overall_lengths       = output["overall_lengths"]
overall_words         = output["overall_words"]
overall_booleans      = output["overall_booleans"]
overall_page_lengths  = output["overall_page_lengths"]
overall_page_words    = output["overall_page_words"]
overall_intro_lengths = output["overall_intro_lengths"]

In [41]:
# Import relevant revision IDs for us
wiki_pages = pd.read_csv("metadata/demonym_case_study_rev_ids_dec312025.txt", sep="\t",header=None)
wiki_pages.columns = ["page_name","revision_id"]

In [44]:
def reduce_dict(d_, k_):
    '''
    Reduce a dictionary to just the relevant keys.
    Input: Dict, list of keys to preserve
    Output: Same dictionary, but reduced
    '''

    d = {k: v for k, v in d_.items() if k in k_}
    return d

In [47]:
# Reduce to the observations for this comparison
page_summary          = reduce_dict(page_summary, wiki_pages["revision_id"].values)
overall_summary       = reduce_dict(overall_summary, wiki_pages["revision_id"].values)
overall_levels        = reduce_dict(overall_levels, wiki_pages["revision_id"].values)
overall_titles        = reduce_dict(overall_titles, wiki_pages["revision_id"].values)
overall_lengths       = reduce_dict(overall_lengths, wiki_pages["revision_id"].values)
overall_words         = reduce_dict(overall_words, wiki_pages["revision_id"].values)
overall_booleans      = reduce_dict(overall_booleans, wiki_pages["revision_id"].values)
overall_page_lengths  = reduce_dict(overall_page_lengths, wiki_pages["revision_id"].values)
overall_page_words    = reduce_dict(overall_page_words, wiki_pages["revision_id"].values)
overall_intro_lengths = reduce_dict(overall_intro_lengths, wiki_pages["revision_id"].values)

### Compare page lengths and introductions

Baseline measures

In [359]:
grok_files = os.listdir(grok_page_dir)
page_lengths_wiki = dict()
page_lengths_grok = dict()
wiki_longer = list()

# Booleans of whether a page contains genetics keywords in introduction plaintext
grok_intro_genetics = dict()
wiki_intro_genetics = dict()

# Regex for keywords
combined = re.compile(
    r'admix|autosom|biobank|chromosom|\bgenes?\b|genetic|genom|genotyp|haplo|mitochon|mt-DNA|mtDNA|PCA|principal component|x[ -]chromosom|y[ -]chromosom|y-DNA|yDNA',
    re.IGNORECASE
)
dna_pattern = re.compile(r'\bDNA\b')

for page in os.listdir(wiki_page_dir):
    page_name = page.split(".txt")[0]

    if page in grok_files:
        # Import Wikipedia and Grokipedia data
        with open(os.path.join(wiki_page_dir, page)) as f:
            wiki_text = re.sub(r'\n\s*\n+', '\n',f.read())
        f.close()

        with open(os.path.join(grok_page_dir, page)) as f:
            grok_text = re.sub(r'\n\s*\n+', '\n',f.read())
        f.close()

        page_lengths_wiki[page_name] = len(wiki_text)
        page_lengths_grok[page_name] = len(grok_text)

        if page_lengths_wiki[page_name] > page_lengths_grok[page_name]:
            wiki_longer = True
        else:
            wiki_longer = False

        # Check for genetics terms in intro
        wiki_intro = wiki_text.split("==")[0]
        grok_intro = grok_text.split("==")[2]

        if combined.search(wiki_intro) or dna_pattern.search(wiki_intro):
            wiki_intro_genetics[page_name] = True
        else:
            wiki_intro_genetics[page_name] = False

        if combined.search(grok_intro) or dna_pattern.search(grok_intro):
            grok_intro_genetics[page_name] = True
        else:
            grok_intro_genetics[page_name] = False


### Compare genetics sections

First we need to calculate the lengths of genetics sections.

In [59]:
pat_genetics_section = r'DNA|(?i:genet)|(?i:admix)|(?i:biobank)|(?i:genom)|(?i:chromosom)|(?i:autosom)|(?i:genotyp)|(?i:mitochon)|(?i:haplo)'

In [66]:
grok_parsed_dir = "data/grokipedia/all_parsed"

In [191]:
# For each Grokipedia page we need to indicate if each section is a genetics section
os.listdir(grok_parsed_dir)

grok_levels = defaultdict(list)
grok_lengths = defaultdict(list)
grok_genetics_idx = defaultdict(list)
grok_pages_genetics = list() # grok pages with genetics sections

grok_page_names = list()

for page_file in os.listdir(grok_parsed_dir):
    with open(os.path.join(grok_parsed_dir, page_file),"r") as fi:
        json_ = json.loads(fi.read())
        fi.close()
        page_name = page_file.split(".json")[0] # This is stored in the JSON but for consistency w/ wikipedia using filename
        grok_page_names.append(page_name) # Simplify for the next step

    section_idx = 0
    for section in json_["sections"]:
        section["plain_text"] = re.sub(r'\[\d+\]','',section["text"])

        # Level of section
        if section["header"]=="(Introduction)":
            grok_levels[page_name].append(0)
        else:
            grok_levels[page_name].append(int(section["level"].replace("h","")))

        # Length of section's plaintext
        grok_lengths[page_name].append(len(section["plain_text"]))

        # This was originally used in the grokipedia analysis
        
        # Index of genetics sections
        if re.search(pat_genetics_section, section["header"]):
        #if combined.search(section["header"]) or dna_pattern.search(section["header"]):
            grok_genetics_idx[page_name].append(section_idx)
            if page_name not in grok_pages_genetics:
                grok_pages_genetics.append(page_name)

        # Increment section index
        section_idx+= 1

In [281]:
# Calculate Grokipedia genetics section lengths
grok_genetics_section_lengths = dict()
for page_name in grok_page_names:
    grok_genetics_section_lengths[page_name] = sum_genetics_lengths(
        grok_levels[page_name], grok_lengths[page_name], grok_genetics_idx[page_name], double_count=True
    )

### Make the comparisons

In [172]:
wiki_page_names = list(wiki_pages["page_name"].values)

In [173]:
wiki_pages_genetics = list() # Names of pages with genetics sections
wiki_pages_revids = list() # Revision IDs used (for linking

# Fill the lists
for revid in overall_booleans.keys():
    if np.sum(overall_booleans[revid]) > 0:
        wiki_pages_revids.append(revid)
        wiki_pages_genetics.append(wiki_pages[wiki_pages["revision_id"]==revid]["page_name"].values[0])

In [174]:
# First comparison: demonyms that exist on both projects
exclusion_list = ["Ashkenazi_Jews","Black_people","White_people"] # Exclude case studies here

In [283]:
wiki_page_names = [i for i in wiki_page_names if i not in exclusion_list]

In [332]:
# Calculate genetics section lengths for each Wikipedia page
wiki_genetics_section_lengths = dict()
for revid in page_summary.keys():
    page_name = wiki_pages[wiki_pages["revision_id"]==revid]["page_name"].values[0] # Extract page name

    # Get the lengths of genetics sections
    wiki_genetics_section_lengths[page_name] = sum_genetics_lengths(
        overall_levels[revid], overall_lengths[revid], np.flatnonzero(overall_booleans[revid]).tolist() ,double_count=False
    )

In [273]:
# Sets to look at (demonyms only)
# Genetics section on both
# Genetics section on Wikipedia only
# Genetics section on Grokipedia only

# Pages that exist on both sites
shared_pages = list(set(wiki_page_names).intersection(set(grok_page_names)))
shared_pages = [i for i in shared_pages if i not in exclusion_list] # Drop case study pages

# Set up the demonym pages
wiki_pages_genetics_demo = [p for p in wiki_pages_genetics if p in shared_pages]
grok_pages_genetics_demo = [p for p in grok_pages_genetics if p in shared_pages]

wiki_and_grok = set(wiki_pages_genetics_demo).intersection(set(grok_pages_genetics_demo))
wiki_only = set(wiki_pages_genetics_demo) - wiki_and_grok
grok_only = set(grok_pages_genetics_demo) - wiki_and_grok

print(len(wiki_and_grok), "pages with genetics sections on both.")
print(len(wiki_only), "pages with genetics sections only on Wikipedia.")
print(len(grok_only), "pages with genetics sections only on Grokipedia.")

66 pages with genetics sections on both.
2 pages with genetics sections only on Wikipedia.
28 pages with genetics sections only on Grokipedia.


In [337]:
# For pages on both projects, check the lengths of their sections
wiki_longer = list()
grok_longer = list()
for page in wiki_and_grok:
    if wiki_genetics_section_lengths[page] > grok_genetics_section_lengths[page]:
        wiki_longer.append(page)
    else:
        grok_longer.append(page)

In [351]:
# Export for viz
out_dir = "output/corpus_comparisons/genetics_sections"
with open(os.path.join(out_dir, "genetics_sections.csv"),"w") as f:
    f.write(",".join(["page","wiki_length","grok_length"]))
    f.write("\n")
    for page in wiki_and_grok:
        f.write(",".join([page,str(wiki_genetics_section_lengths[page]),str(grok_genetics_section_lengths[page])]))
        f.write("\n")
    for page in grok_only:
        f.write(",".join([page,str(0),str(grok_genetics_section_lengths[page])]))
        f.write("\n")
    for page in wiki_only:
        f.write(",".join([page,str(wiki_genetics_section_lengths[page]),str(0)]))
        f.write("\n")
f.close()